# Kernels and carryover

A dose-response kernel is an *expression builder*: it declares its parameters — **scales**
carrying dose or outcome dimensions, **shapes** dimensionless — and returns `axiom.core` nodes
for the response, the dimensionless saturation, and the closed-form derivative. Carryover is a
normalized FIR weight vector, also built from nodes, applied with `Convolve`. Nothing here is a
numpy transform: every number comes from the interpreters (rule 3, "one `forward()`").

In [ ]:
import numpy as np

from axiom.core import D, Data, dimension, dimensionless, latex, value
from axiom.surface import (
    CARRYOVERS, KERNELS, AnyCarryover, AnyKernel, CarryoverKernel, CarryoverRole, DelayedCarryover,
    ExponentialKernel, GaussianProcessKernel, GeometricCarryover, HillKernel, KernelRole, LinearKernel, LogisticKernel,
    NoCarryover, PiecewiseLinearKernel, PolynomialKernel, PowerKernel, ResponseKernel,
    SplineKernel, WeibullCarryover, carryover_from_name, kernel_from_name,
)
from axiom.surface.kernels import (
    AMPLITUDE_PRIOR_FAMILIES, BASIS_PRIOR_FAMILIES, CovarianceFamily,
)

In [ ]:
print(sorted(KERNELS), "|", sorted(CARRYOVERS))
hill: ResponseKernel = HillKernel(reference_dose=50.0, amplitude_scale=10.0)
dose = Data(name="a", dimension=D.currency)
for p in hill.parameters("a", D.currency, D.outcome):
    role: KernelRole = hill.roles[p.name.rsplit("_", 1)[0]] if p.name.rsplit("_", 1)[0] in hill.roles else "shape"
    print(f"{p.name:8s} dim={p.dimension!s:4s} prior={p.prior.family}{p.prior.hyper}")
print("saturating:", hill.saturating, "| roles:", dict(hill.roles))

In [ ]:
resp = hill.response(dose, "a")
print(dimension(resp), "|", dimension(hill.saturation(dose, "a")), "|", dimension(hill.derivative(dose, "a")))
print(latex(hill.saturation(dose, "a")))
theta = {"k_a": 50.0, "s_a": 2.0, "beta_a": 10.0}
grid = {"a": np.array([0.0, 25.0, 50.0, 100.0, 200.0])}
print(value(resp, data=grid, params=theta).round(3))
print(value(hill.derivative(dose, "a"), data=grid, params=theta).round(4))

## Unit invariance (exit criterion 6)

Express the same world in another unit: doses and the scale `k` multiplied by the same factor,
shapes untouched. The response is unchanged. This is what lets shape parameters pool across
studies in different currencies or time grids.

In [ ]:
cents = {"a": grid["a"] * 100}
theta_cents = {**theta, "k_a": theta["k_a"] * 100}
print(np.allclose(value(resp, data=cents, params=theta_cents), value(resp, data=grid, params=theta)))

## The families

In [ ]:
for fam in (LogisticKernel(reference_dose=50.0), ExponentialKernel(reference_dose=50.0), PowerKernel(reference_dose=50.0), LinearKernel(reference_dose=50.0)):
    k: AnyKernel = fam
    names = [p.name for p in k.parameters("a", D.currency, D.outcome)]
    print(f"{k.name:12s} saturating={k.saturating!s:5s} params={names}")
print(kernel_from_name("hill", reference_dose=10.0))

## The basis families: when the response bends back

`hill`, `logistic`, `exponential`, `power` and `linear` are all **monotone**: more dose,
more response. A dose that helps and then, past some level, starts to hurt is outside
every one of them, and a fit will say so by leaving a large residual that is systematic
in dose rather than by warning you.

The three **basis** families exist for that case. Each writes the response as a signed
sum over a fixed basis of the dimensionless coordinate `u = dose / reference_dose`:

| family | basis | good for |
|---|---|---|
| `polynomial` | `u, u², …, u^degree` | a single smooth bend, few dose levels |
| `spline` | natural cubic on fixed knots | several bends, **linear** extrapolation |
| `piecewise_linear` | `u, (u−t₁)₊, …` | a slope that changes at a stated dose |

They differ from the saturating families in three ways that the rest of `axiom` reads off
`roles`: they declare **one coefficient per basis function** instead of one amplitude,
those coefficients are **signed**, and the response is **linear in every one of them**.

In [ ]:
basis = (
    PolynomialKernel(reference_dose=40.0, degree=3),
    SplineKernel(reference_dose=40.0, knots=(10.0, 20.0, 30.0)),
    PiecewiseLinearKernel(reference_dose=40.0, knots=(15.0, 25.0)),
)
for fam in basis:
    k: AnyKernel = fam
    names = [p.name for p in k.parameters("a", D.currency, D.outcome)]
    roles: dict[str, KernelRole] = dict(k.roles)
    print(f"{k.name:17s} saturating={k.saturating!s:5s} params={names}")
    print(f"{'':17s} roles={set(roles.values())}")
print("\ncoefficient priors may be signed:", sorted(BASIS_PRIOR_FAMILIES))
print("the amplitude of a saturating family may not:", sorted(AMPLITUDE_PRIOR_FAMILIES))

`reference_dose` is not a parameter for these families, it is the coordinate. Set it to
the **top of the dose range you mean to fit** so `u` lands in `[0, 1]` and one prior scale
is sensible for every basis function; knots are given in dose units and divided by it.

Now the property the saturating families cannot have.

In [ ]:
grid = {"a": np.linspace(0.0, 48.0, 13)}
poly = PolynomialKernel(reference_dose=40.0, degree=3)
theta = {"beta1_a": 9.0, "beta2_a": -16.0, "beta3_a": 8.0}
response = value(poly.response(dose, "a"), data=grid, params=theta)
slope = value(poly.derivative(dose, "a"), data=grid, params=theta)
print("dose    ", " ".join(f"{d:6.0f}" for d in grid["a"]))
print("response", " ".join(f"{v:6.2f}" for v in np.ravel(response)))
print("d/d dose", " ".join(f"{v:6.2f}" for v in np.ravel(slope)))
print("\nrises then falls:", bool(np.any(np.diff(np.ravel(response)) > 0) and np.any(np.diff(np.ravel(response)) < 0)))
print("dimensions:", dimension(poly.response(dose, "a")), "|", dimension(poly.derivative(dose, "a")))

### Linear in every coefficient
`parameter_roles` reports each coefficient as `linear`, because it is: the response is a
matrix product of a fixed basis with the coefficient vector. That is not a cosmetic
label — it is what makes `surface.linearize` exact rather than a local approximation, and
what lets the alphabet-optimality criteria in `surface.designs` mean what they say.

In [ ]:
one_at_a_time = np.zeros_like(np.ravel(response))
for stem in poly.roles:
    only = {k: (v if k == f"{stem}_a" else 0.0) for k, v in theta.items()}
    one_at_a_time = one_at_a_time + np.ravel(value(poly.response(dose, "a"), data=grid, params=only))
print("additive over the coefficient vector:", np.allclose(one_at_a_time, np.ravel(response)))
doubled = value(poly.response(dose, "a"), data=grid, params={k: 2 * v for k, v in theta.items()})
print("homogeneous in it:                   ", np.allclose(np.ravel(doubled), 2 * np.ravel(response)))

### The natural cubic spline extrapolates like a line
A cubic polynomial does whatever its leading term says once you leave the fitted range.
A *natural* cubic spline is constrained to be linear outside its boundary knots, which is
the difference between an extrapolation you can show someone and one you cannot.

In [ ]:
spline = SplineKernel(reference_dose=40.0, knots=(8.0, 16.0, 24.0, 32.0))
theta_s = {"beta1_a": 6.0, "beta2_a": -22.0, "beta3_a": 16.0}
outside = {"a": np.array([34.0, 38.0, 42.0, 46.0])}
curve = np.ravel(value(spline.response(dose, "a"), data=outside, params=theta_s))
print("spline beyond the last knot:", curve.round(3), "-> second difference",
      np.round(np.diff(curve, 2), 9))
cubic = np.ravel(value(poly.response(dose, "a"), data=outside, params=theta))
print("cubic  beyond the last knot:", cubic.round(3), "-> second difference",
      np.round(np.diff(cubic, 2), 9))

### A piecewise-linear fit reads as a sentence
`beta1_a` is the slope of the first segment and every later coefficient is the **change**
in slope at its knot, so the fitted vector says "the response per unit dose was this, and
at 15 it changed by that". `step(0) = 0`, so a derivative read exactly *at* a knot reports
the slope arriving into it.

In [ ]:
broken = PiecewiseLinearKernel(reference_dose=40.0, knots=(15.0, 25.0))
theta_b = {"beta1_a": 8.0, "beta2_a": -14.0, "beta3_a": 4.0}
at = {"a": np.array([5.0, 15.0, 20.0, 30.0])}
print("slope by dose:", np.ravel(value(broken.derivative(dose, "a"), data=at, params=theta_b)).round(4))
# 5: before both knots; 15: exactly on the first, so still the arriving slope;
# 20: after the first; 30: after both.
print("expected     :", np.round(np.array([8.0, 8.0, 8.0 - 14.0, 8.0 - 14.0 + 4.0]) / 40.0, 4))
print("\nsaturation_derivative is on every family, basis or not:")
for fam in (HillKernel(reference_dose=50.0), broken):
    print(" ", fam.name, latex(fam.saturation_derivative(dose, "a"))[:72], "...")

## A Gaussian process, when you will not commit to a shape at all

`HillKernel` assumes saturation. `SplineKernel` assumes a knot set. `GaussianProcessKernel`
assumes only **stationarity and a smoothness scale**: it puts a GP prior on the
dose-response and estimates the lengthscale from the data.

The exact GP would need a multivariate normal over the latent function, which is not what
`core.Likelihood` evaluates. What this ships is the Hilbert-space reduced-rank
construction (Solin & Särkkä 2020): on a bounded interval the Laplacian eigenfunctions are
a fixed sine basis, and a stationary GP is recovered by giving basis `j` the prior standard
deviation `sqrt(S(w_j))`, with `S` the covariance's spectral density. So it is, structurally,
one more basis family — with the coefficient scales tied together by two interpretable
hyperparameters instead of being free.

In [ ]:
gp = GaussianProcessKernel(reference_dose=40.0, amplitude_scale=8.0)
covariance: CovarianceFamily = gp.covariance
print(f"covariance          {covariance}")
print(f"basis functions     {gp.n_basis}   boundary L = {gp.boundary} (factor {gp.boundary_factor})")
print(f"parameters          {len(gp.stems)}: {gp.stems[:3]} ... {gp.stems[-1]}")
print(f"roles               {sorted(set(gp.roles.values()))}")
print(f"amplitudes          {[s for s, r in gp.roles.items() if r == 'amplitude']}")
print("\nUnlike the basis families it has exactly one amplitude -- the sign lives in the")
print("z coefficients -- so response == beta * saturation still holds node for node.")

### The two numbers that decide whether it is a GP

`n_basis` sets how *short* a lengthscale can be represented and `boundary_factor` how
*long*. They trade against each other, and getting them wrong does not raise — it quietly
fits a different prior than the one you asked for. `covariance_error` compares the
covariance the basis actually implies against the exact one and returns the worst gap, so
the question has a number attached.

In [ ]:
print(f"{'lengthscale':>12} {'cov error':>10} {'usable':>8}")
for ell in (0.03, 0.08, 0.10, 0.30, 0.70, 1.20):
    print(f"{ell:12.2f} {gp.covariance_error(ell):10.4f} {str(gp.sufficient_for(ell)):>8}")
print("\nmore basis functions buy the short end; a wider boundary buys the long end:")
print("  n_basis=96          ell=0.03 ->", GaussianProcessKernel(n_basis=96).sufficient_for(0.03))
print("  boundary_factor=8   ell=1.20 ->",
      GaussianProcessKernel(n_basis=96, boundary_factor=8.0).sufficient_for(1.2))

### What it looks like
Draws from the prior at two lengthscales, and the property that makes them a GP: the
implied covariance really is the one being approximated.

In [ ]:
rng = np.random.default_rng(0)
doses = np.linspace(0.0, 40.0, 9)
for ell in (0.12, 0.45):
    theta = {"ell_a": ell, "beta_a": 8.0}
    theta |= {f"z{j}_a": float(rng.normal()) for j in range(1, gp.n_basis + 1)}
    drawn = np.ravel(value(gp.response(dose, "a"), data={"a": doses}, params=theta))
    print(f"ell={ell:4.2f}  f(dose) = " + " ".join(f"{v:6.2f}" for v in drawn))
    print(f"{'':10s} f(0) = {drawn[0]:.1e} exactly, so the surface intercept keeps its meaning")

u = np.linspace(0.0, 1.0, 5)
print("\nimplied covariance at ell=0.3 (rows) against the exact squared exponential:")
print(np.round(gp.implied_covariance(0.3, u), 3))
print(np.round(gp.exact_covariance(0.3, u), 3))

## Carryover

Weights are a dimensionless expression (`Pow`, `Reduce(keepdims=True)`, `Div`), so the jax
interpreter sees them and NUTS can sample their parameters. A unit impulse through `apply`
reproduces the weights; `half_life` is a diagnostic on parameter values.

In [ ]:
geo: CarryoverKernel = GeometricCarryover(max_lag=6)
w = geo.weights("a")
print(dimension(w), [p.name for p in geo.parameters("a")])
impulse = {"a": np.array([1.0, 0, 0, 0, 0, 0, 0, 0])}
print(value(geo.apply(dose, "a"), data=impulse, params={"lam_a": 0.6}).round(4))
print("half-life:", geo.half_life({"lam_a": 0.6}, "a"))
role: CarryoverRole = "shape"

In [ ]:
for c in (DelayedCarryover(max_lag=6), WeibullCarryover(max_lag=6), NoCarryover()):
    cc: AnyCarryover = c
    print(cc.name, [p.name for p in cc.parameters("a")])
delayed = DelayedCarryover(max_lag=8)
print(value(delayed.weights("a"), params={"lam_a": 0.5, "theta_a": 3.0}).round(3))
print(carryover_from_name("geometric", max_lag=4))

Draw-shaped parameters broadcast: a `(draws, 1)` array of `lam` gives one normalized weight
row per draw — the normalization keeps the lag axis (`Reduce(keepdims=True)`).

In [ ]:
value(geo.weights("a"), params={"lam_a": np.array([[0.2], [0.5], [0.9]])}).round(3)